In this notebook, we demonstrate how to extract features from Magnetoencephalography (MEG) recordings. To do so, we use the `mainParallel.py` module. This function can be run both in parallel and sequentially. Here, we show how to work with both options.

Feature extraction involves several steps, including bad channel detection, preprocessing, power spectral density (PSD) computation, periodic power isolation, feature extraction, and summarization across the brain. All these steps are embedded within `mainParallel.py`.

**Note**: It is necessary that the recording names follow the BIDS template.

## Parameters to Specify

- **`datasets`**  
  A nested Python dictionary where each key corresponds to a dataset name (e.g., `'BTNRH'`, `'CAMCAN'`, etc.). Each value is a dictionary with the following fields:

  - **`base_dir`**:  
    The base directory path for the dataset, typically the root folder containing the subject data.

  - **`task`**:  
    The task label used in the filenames (e.g., `'task-rest'` in `sub-xx_ses-02_task-rest_run-02_meg.ds`). Note that files must be named according to the BIDS format.

  - **`ending`**:  
    The filename suffix, including its extension (e.g., `'meg.fif'`, `'meg.ds'`, or `'4-Restin/4D'`). Note that files must be named according to the BIDS format.

  This structure ensures compatibility across various MEG hardware systems.

- **`project_dir`**  
  The directory where all output files will be saved.  
  **Note:** Use a consistent path across all notebooks to avoid confusion or file overwriting.

- **`conda_environment_name`**  
  The name of the Python environment used to run the jobs.  
  This environment must have the **MEGaNorm** package installed.

- **`slurm_username`**  
  If using SLURM for parallel computation, provide your SLURM `username`.  
  This enables job monitoring and automatic resubmission of incomplete tasks.

In [1]:
datasets = {
    'BTNRH': {
        'base_dir': ...,
        'task': ...,
        "ending" : ...
        },
    'CAMCAN': {
        'base_dir': ...,
        'task': ...,
        "ending" : ...
        },
    }

project_dir = ...
conda_environment_name = ...
slurm_username = ...

# Packages

In [10]:
import meganorm
import json
import os
import warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# === meganorm: I/O utilities ===
from meganorm.utils.IO import (
    merge_datasets_with_glob,
)

# for parallel feature extraction
from meganorm.utils.parallel import (
    collect_results,
    auto_parallel_feature_extraction
)

# for sequential feature extraction
from meganorm.src.mainParallel import main

Now we set some paths to save the results. We then use `merge_datasets_with_glob` to store the MEG paths for each subject in a dictionary. If a subject has multiple recordings, only the first one will be used in the analysis.

In [5]:
features_dir = os.path.join(project_dir, 'Features')
features_log_path = os.path.join(features_dir, 'log')
features_temp_path = os.path.join(features_dir,'temp')

# configs for parallel feature extraction
job_configs = {'log_path':features_log_path, 'module':conda_environment_name, 'time':'1:00:00', 'memory':'20GB', 
                'partition':'normal', 'core':1, 'node':1, 'batch_file_name':'batch_job'}

# a log file to save logs of feature extraction
if not os.path.isdir(features_log_path):
    os.makedirs(features_log_path)

# a directory to save extracted features temporarily
if not os.path.isdir(features_temp_path):
    os.makedirs(features_temp_path)

subjects = merge_datasets_with_glob(datasets)

# Feature extraction config
Now we need to set the config for feature extraction. This function make a config file that determine which parameters should be used for each step. You can change the default at your opinion. Currently, our package supports extracting the following feature categories:

* Offset
* Exponent
* Peak_Center
* Peak_Power
* Peak_Width
* Adjusted_Canonical_Relative_Power 
* Adjusted_Canonical_Absolute_Power
* Adjusted_Individualized_Relative_Power
* Adjusted_Individualized_Absolute_Power
* OriginalPSD_Canonical_Relative_Power 
* OriginalPSD_Canonical_Absolute_Power
* OriginalPSD_Individualized_Relative_Power
* OriginalPSD_Individualized_Absolute_Power


> You can set any of them to `True` to extract the corresponding feature. As an example, here we only set `Adjusted_Canonical_Relative_Power` to `True`.


In [6]:
def make_config(path=None):
    """
    Create a configuration dictionary for a neuroimaging preprocessing pipeline.

    This function generates configuration settings for preprocessing, feature extraction,
    spectral analysis, and other relevant parameters used in processing EEG/MEG data.
    Optionally, it saves the generated configuration to a JSON file in the specified path.

    Parameters
    ----------
    path : str, optional
        The directory path where the configuration file should be saved. If not provided,
        the configuration is not saved to a file.

    Returns
    -------
    config : dict
        The configuration dictionary containing settings for preprocessing, feature extraction,
        and analysis.

    Notes
    -----
    - The generated configuration includes settings for ICA preprocessing, spectral
      estimation, and feature extraction for EEG/MEG data.
    - Default values are provided for the majority of settings.
    - If `path` is provided, a `.json` file containing the configuration will be saved.
    """

    # preprocess configurations =================================================
    # downsample data
    config = dict()

    # You could also set layout to None to have high 
    # choices: all, lobe, None
    config["which_layout"] = "all"

    # which sensor type should be used
    # choices: meg, mag, grad, eeg, opm
    config["which_sensor"] = "meg"
    # config['fs'] = 1000

    # ICA configuration
    config['ica_n_component'] = 30
    config['ica_max_iter'] = 800
    config['ica_method'] = "fastica"

    # lower and upper cutoff frequencies in a bandpass filter
    config['cutoffFreqLow'] = 1
    config['cutoffFreqHigh'] = 45

    config["resampling_rate"] = 1000
    config["digital_filter"] = True
    config["notch_filter"] = False

    config["apply_ica"] = True

    config["auto_ica_corr_thr"] = 0.9

    # options are "average", "REST", and None 
    config["rereference_method"]= "average"

    # variance threshold across time
    config["mag_var_threshold"] = 4e-12
    config["grad_var_threshold"] = 4000e-13
    config["eeg_var_threshold"] = 40e-6
    # flatness threshold across time
    config["mag_flat_threshold"] = 10e-15
    config["grad_flat_threshold"] = 10e-15
    config["eeg_flat_threshold"] = 40e-6
    # variance thershold across channels
    config["zscore_std_thresh"] = 15 # change this

    # segmentation ==============================================
    #start time of the raw data to use in seconds, this is to avoid possible eye blinks in close-eyed resting state. 
    config['segments_tmin'] = 20
    # end time of the raw data to use in seconds, this is to avoid possible eye blinks in close-eyed resting state.
    config['segments_tmax'] = -20
    # length of MEG segments in seconds
    config['segments_length'] = 10
    # amount of overlap between MEG sigals in seconds
    config['segments_overlap'] = 2

    # PSD ==============================================
    # Spectral estimation method
    config['psd_method'] = "welch"
    # amount of overlap between windows in Welch's method
    config['psd_n_overlap'] = 1
    config['psd_n_fft'] = 2
    # number of samples in psd
    config["psd_n_per_seg"] = 2

    # fooof analysis configurations ==============================================
    # Desired frequency range to run FOOOF
    config['fooof_freq_range_low'] = 3
    config['fooof_freq_range_high'] = 40
    config["fooof_freq_range_low"] = 3
    config["fooof_freq_range_high"] = 40
    # which mode should be used for fitting; choices (knee, fixed)
    config["aperiodic_mode"] = "knee"
    # minimum acceptable peak width in fooof analysis
    config["fooof_peak_width_limits"] = [1.0, 12.0]
    #Absolute threshold for detecting peaks
    config['fooof_min_peak_height'] = 0
    #Relative threshold for detecting peaks
    config['fooof_peak_threshold'] = 2

    # feature extraction ==========================================================
    # Define frequency bands
    config['freq_bands'] = {
                            'Theta': (3, 8),
                            'Alpha': (8, 13),
                            'Beta': (13, 30),
                            'Gamma': (30, 40),
                            # 'Broadband': (3, 40)
                            }

    # Define individualized frequency range over main peaks in each freq band
    config['individualized_band_ranges'] = { 
                                            'Theta': (-2, 3),
                                            'Alpha': (-2, 3), # change to (-4,2)
                                            'Beta': (-8, 9),
                                            'Gamma': (-5, 5)
                                            }

    # least acceptable R squred of fitted models
    config['min_r_squared'] = 0.9 
 
    config['feature_categories'] = {
                                    "Offset":False,
                                    "Exponent":False,
                                    "Peak_Center":False,
                                    "Peak_Power":False,
                                    "Peak_Width":False,
                                    "Adjusted_Canonical_Relative_Power":True, 
                                    "Adjusted_Canonical_Absolute_Power":False,
                                    "Adjusted_Individualized_Relative_Power":False,
                                    "Adjusted_Individualized_Absolute_Power":False,
                                    "OriginalPSD_Canonical_Relative_Power":False, 
                                    "OriginalPSD_Canonical_Absolute_Power":False,
                                    "OriginalPSD_Individualized_Relative_Power":False,
                                    "OriginalPSD_Individualized_Absolute_Power":False,
                                    }
    
    config["fooof_res_save_path"] = None

    config["random_state"] = 42

    if path is not None:
        out_file = open(os.path.join(path, "configuration.json"), "w") 
        json.dump(config, out_file, indent = 6) 
        out_file.close()

    return config 

In [7]:
# Create a config file that contains all parameters for feature extraction,  
# including filtering settings, which features to extract, and other options.
configs = make_config(project_dir)

# Parallel feature extraction using Slurm

In [ ]:
auto_parallel_feature_extraction(
    mainParallel_path=os.path.abspath(meganorm.src.mainParallel.__file__), # mainParallel.py's path
    features_dir=features_dir, # Where to save the extracted features
    subjects=subjects, # a dictionary containing subjects' ID and their MEG paths
    job_configs=job_configs, # slurm job configs
    config_file=os.path.join(project_dir, 'configuration.json'), # The feature extraction configuratiob file
    username=slurm_username, # environment name 
    auto_rerun=True, # If failed for a subject, run again
    auto_collect=True, # Automatically collect each subject's CSV file and combine them into a single DataFrame
    max_try=3) # If jobs get failed for a subject, try only three times

# Sequential feature extraction
Here we try to extract features sequentially. This may take quite some time, depending on your computational resources and the number of datasets.

In [ ]:
for subject_id, subject_paths in subjects.items(): # For every subject, do the following

    subject_paths = list(filter(lambda x: len(x), subject_paths.split("*")))
    subject_path = subject_paths[0] # Choose the first MEG recording

    main(*[
        subject_path, # Subject's MEG path
        features_temp_path, # Where to save
        subject_id, # subject ID
        "--configs", 
        os.path.join(project_dir, "configuration.json") # feature extraction config file
    ])

# collect each subject's CSV file and combine them into a single DataFrame
collect_results(features_dir, subjects, features_temp_path, file_name='all_features', clean=False) 

Congrats! You now have a CSV file at your disposal.